In [163]:
##### Compiles csv of training data for regions in dataset and does final cleaning

import os
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from functools import reduce

In [164]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent 

# folder of sub-national predictors 
predictors = Path(f'{cd}/Data/Clean/Predictors/Vectors')

# country averages
country_avg = pd.read_csv(f'{cd}/Data/Clean/Predictors/country_averages.csv')

# intensities 
capital = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_capital_intensity.csv")
labor = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_labor_intensity.csv")

# lat / lon 
lat_lon = pd.read_csv(f"{cd}/Data/Clean/Predictors/Vectors/lat_lon.csv")

# save path
save_path_capital_abs = f'{cd}/Data/Clean/Training_data/capital_absolute.csv'
save_path_labor_abs = f'{cd}/Data/Clean/Training_data/labor_absolute.csv'

save_path_capital_rtv = f'{cd}/Data/Clean/Training_data/capital_relative.csv'
save_path_labor_rtv = f'{cd}/Data/Clean/Training_data/labor_relative.csv'

### Assemble vectors for absolute training data

In [165]:
##### Merge all predictors into one df (dropping all duplicate PROJ_ID entries first)
dfs = []
for f in os.listdir(predictors):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join(predictors, f))
        dupes = df['PROJ_ID'].duplicated().sum()
        if dupes > 0:
            df = df.drop_duplicates(subset='PROJ_ID')
        dfs.append(df)

predictors_df = reduce(lambda left, right: pd.merge(left, right, on='PROJ_ID', how='outer'), dfs)

In [166]:
### Merge with capital and labor 

### Capital
capital = capital.drop_duplicates(subset='PROJ_ID')
capital_merge = capital.merge(predictors_df, on='PROJ_ID', how='left')

### Labor
labor = labor.drop_duplicates(subset='PROJ_ID')
labor_merge = labor.merge(predictors_df, on='PROJ_ID', how='left')

In [167]:
### Save
capital_merge.to_csv(save_path_capital_abs, index=False)
labor_merge.to_csv(save_path_labor_abs, index=False)

### Assemble vectors for relative training data

In [168]:
##### Merge with country averages 

# Add country ID column
capital_merge['ISO3'] = capital_merge['PROJ_ID'].str[:3]
labor_merge['ISO3'] = labor_merge['PROJ_ID'].str[:3]

# Merge with country data
country_avg = country_avg.rename(columns=lambda c: c if c == 'ISO3' else f'country_{c}')

capital_merge_country = capital_merge.merge(country_avg, on='ISO3', how='left')
labor_merge_country = labor_merge.merge(country_avg, on='ISO3', how='left')


In [169]:
##### Do log transformations of all intensities and densities

log_columns_cap = ['capital_intensity_USD_per_USD', 'capital_intensity_USD_per_tonne',
       'pop_density_people_per_km2', 'country_pop_density_people_per_km2', 'USD_production_per_HA', 'country_USD_production_per_HA',
       'tonnes_production_per_HA', 'country_tonnes_production_per_HA', 'buffalo_density_per_km2', 'country_buffalo_density_per_km2',
       'chicken_density_per_km2', 'country_chicken_density_per_km2', 'cattle_density_per_km2', 'country_cattle_density_per_km2',
       'goats_density_per_km2', 'country_goats_density_per_km2', 'pigs_density_per_km2', 'country_pigs_density_per_km2',
       'sheep_density_per_km2', 'country_sheep_density_per_km2', 'livestock_density_LU_per_km2', 'country_livestock_density_LU_per_km2',
       'country_labor_intensity_jobs_per_million_USD',
       'country_capital_intensity_USD_per_USD',
       'country_labor_intensity_jobs_per_tonne',
       'country_capital_intensity_USD_per_tonne']

log_columns_lab = ['labor_intensity_jobs_per_million_USD', 'labor_intensity_jobs_per_tonne',
       'pop_density_people_per_km2', 'country_pop_density_people_per_km2', 'USD_production_per_HA', 'country_USD_production_per_HA',
       'tonnes_production_per_HA', 'country_tonnes_production_per_HA', 'buffalo_density_per_km2', 'country_buffalo_density_per_km2',
       'chicken_density_per_km2', 'country_chicken_density_per_km2', 'cattle_density_per_km2', 'country_cattle_density_per_km2',
       'goats_density_per_km2', 'country_goats_density_per_km2', 'pigs_density_per_km2', 'country_pigs_density_per_km2',
       'sheep_density_per_km2', 'country_sheep_density_per_km2', 'livestock_density_LU_per_km2', 'country_livestock_density_LU_per_km2',
       'country_labor_intensity_jobs_per_million_USD',
       'country_capital_intensity_USD_per_USD',
       'country_labor_intensity_jobs_per_tonne',
       'country_capital_intensity_USD_per_tonne']

for col in log_columns_cap:
    capital_merge_country[f"log_{col}"] = np.log(capital_merge_country[col] + 1e-12)
    capital_merge_country = capital_merge_country.drop(columns=[col])

for col in log_columns_lab:
    labor_merge_country[f"log_{col}"] = np.log(labor_merge_country[col] + 1e-12)
    labor_merge_country = labor_merge_country.drop(columns=[col])

In [170]:
##### Calculate relative variables 

# log variables 
log_columns_cap = ['capital_intensity_USD_per_USD', 'capital_intensity_USD_per_tonne',
       'pop_density_people_per_km2', 'USD_production_per_HA',
       'tonnes_production_per_HA', 'buffalo_density_per_km2',
       'chicken_density_per_km2', 'cattle_density_per_km2',
       'goats_density_per_km2', 'pigs_density_per_km2',
       'sheep_density_per_km2', 'livestock_density_LU_per_km2']

log_columns_lab = ['labor_intensity_jobs_per_million_USD', 'labor_intensity_jobs_per_tonne',
       'pop_density_people_per_km2', 'USD_production_per_HA',
       'tonnes_production_per_HA', 'buffalo_density_per_km2',
       'chicken_density_per_km2', 'cattle_density_per_km2',
       'goats_density_per_km2', 'pigs_density_per_km2',
       'sheep_density_per_km2', 'livestock_density_LU_per_km2']

for col in log_columns_cap:
    capital_merge_country[f"rtv_log_{col}"] = capital_merge_country[f'log_{col}'] - capital_merge_country[f"log_country_{col}"]
    capital_merge_country = capital_merge_country.drop(columns=[f'log_{col}', f"log_country_{col}"])

for col in log_columns_lab:
    labor_merge_country[f"rtv_log_{col}"] = labor_merge_country[f'log_{col}'] - labor_merge_country[f"log_country_{col}"]
    labor_merge_country = labor_merge_country.drop(columns=[f'log_{col}', f"log_country_{col}"])

# other variables 
non_log_var = ['SOC', 'pct_cropland_irrigated', 'female_share','average_travel_time_city',
       'average_travel_time_port', 'probability_economic_land_use_objective',
       'probability_survival_land_use_objective', 'growing_season_length_days',
       'child_dependency_ratio', 'GDP_pc', 'slope', 
       'cereals_share_production_USD', 'fibres_share_production_USD',
       'fruits_share_production_USD', 'oilcrops_share_production_USD',
       'pulses_share_production_USD', 'roots_tubers_share_production_USD',
       'rest_of_crops_share_production_USD',
       'sugar_crops_share_production_USD', 'vegetables_share_production_USD',
       'rubber_share_production_USD', 'ruminants_share_production_USD',
       'monogastrics_share_production_USD', 'poultry_share_production_USD',
       'other_share_production_USD', 'pct_GDP_ag', 'share_vlarge_field',
       'share_large_field', 'share_medium_field', 'share_small_field',
       'share_vsmall_field', 'share_with_nightlights',
       'crop_intensity']

for col in non_log_var:
    capital_merge_country[f"rtv_{col}"] = capital_merge_country[col] / capital_merge_country[f"country_{col}"]
    capital_merge_country = capital_merge_country.drop(columns=[col, f"country_{col}"])

for col in non_log_var:
    labor_merge_country[f"rtv_{col}"] = labor_merge_country[col] / labor_merge_country[f"country_{col}"]
    labor_merge_country = labor_merge_country.drop(columns=[col, f"country_{col}"])

# drop unneeeded cols
col_to_drop_cap = ['ag_capital_stock_USD_nominal','total_production_USD', 'total_production_t','lon', 'lat', 'ISO3', 'log_country_labor_intensity_jobs_per_million_USD', 'log_country_labor_intensity_jobs_per_tonne']
col_to_drop_lab = ['ag_jobs','total_production_USD', 'total_production_t','lon', 'lat', 'ISO3', 'log_country_capital_intensity_USD_per_USD', 'log_country_capital_intensity_USD_per_tonne']

capital_merge_country = capital_merge_country.drop(columns=col_to_drop_cap)
labor_merge_country = labor_merge_country.drop(columns=col_to_drop_lab)


In [171]:
### Save
capital_merge_country.to_csv(save_path_capital_rtv, index=False)
labor_merge_country.to_csv(save_path_labor_rtv, index=False)